In [ ]:
import json
import math
import pathlib
import re

import geopy.distance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:

project_root = pathlib.Path('../../..')
project_root.resolve()

In [ ]:
metroflex_data = project_root / 'data/metroflex'
metroflex_data.resolve()

In [ ]:
output_dir = project_root / 'output/roanoke/metroflex'
output_dir.mkdir(exist_ok=True, parents=True)
output_dir.resolve()

In [ ]:
with open(metroflex_data / 'metroflex-addresses.json', 'r') as f:
    address_locations = json.load(f)
address_locations

In [ ]:
def max_distance(matches):
    dist = 0
    loc1 = matches[0]['location']
    for match in matches[1:]:
        loc2 = match['location']
        dist = max(
            dist,
            geopy.distance.geodesic(
                (loc1['lat'], loc1['lng']),
                (loc2['lat'], loc2['lng']),
            ).m
        )
    return dist


def filter_by_state(matches, state):
    return [
        match
        for match in matches
        if match['address_components']['state'] == state
    ]


def filter_by_matching_start(matches, key):
    start = key.lower().split(' ')[0] + ' '
    positive_matches = [
        match
        for match in matches
        if match['formatted_address'].lower().startswith(start)
    ]
    if len(positive_matches) > 0:
        return positive_matches
    return matches


def filter_by_cities(matches, cities):
    return [
        match
        for match in matches
        if match['address_components']['city'] in cities
    ]


def filter_by_matching_suffixes(matches, key, suffix_pairs):
    for sfx, ptx in suffix_pairs:
        if re.search(ptx, key.lower()) is None:
            continue
        new_matches = [
            match
            for match in matches
            if match['address_components'].get('suffix', '').lower() == sfx
        ]
        if len(new_matches) > 0:
            matches = new_matches
    return matches


def collapse_identical_addresses(matches):
    addys = {}
    for match in sorted(matches, key=lambda x: x['accuracy']):
        addys[match['formatted_address']] = match
    return list(addys.values())


def collapse_similar_addresses(matches, parts):
    addys = {}
    for match in sorted(matches, key=lambda x: x['accuracy']):
        key = '-'.join([
            match['address_components'].get(part, '')
            for part in parts
        ])
        addys[key] = match
    return list(addys.values())


def select_most_accurate(matches):
    matches = sorted(matches, key=lambda x: x['accuracy'], reverse=True)
    return matches[0]


def remove_non_matching_streets(matches, key):
    return [
        match for match in matches
        if ' '.join(match['formatted_address'].lower().split(' ')[:2]).startswith(' '.join(key.lower().split(' ')[:2]))
    ]


def lookup(address_locations, key, details, exceptions=None, **kwargs):
    if exceptions is None:
        exceptions = []

    if key not in address_locations:
        return None

    address = address_locations[key]
    matches = address['results']
    matches = collapse_identical_addresses(matches)
    matches = filter_by_state(matches, 'VA')
    if len(matches) == 0:
        return None

    matches = filter_by_matching_start(matches, key)

    if len(matches) > 1:
        matches = filter_by_cities(matches, [
            'Roanoke',
            'Salem',
            'Vinton',
        ])

    if len(matches) == 0:
        return None

    if len(matches) > 1:
        suffix_pairs = [
            # sfx, key_ptx
            ('st', r'\bst\b'),
            ('st', r'\bstreet\b'),
            ('ln', r'\blane\b'),
            ('ln', r'\bln\b'),
            ('rd', r'\broad\b'),
            ('rd', r'\brd\b'),
            ('dr', r'\bdrive\b'),
            ('dr', r'\bdr\b'),
            ('ave', r'\bave\b'),
            ('ave', r'\bavenue\b'),
            ('cir', r'\bcircle\b'),
        ]
        matches = filter_by_matching_suffixes(matches, key, suffix_pairs)

    matches = collapse_similar_addresses(matches, ['number', 'street', 'suffix'])
    if len(matches) > 1:
        max_dist = max_distance(matches)
        if max_dist < 800:
            matches = matches[:1]
        else:
            # last ditch
            matches = remove_non_matching_streets(matches, key)
            if len(matches) == 1:
                return matches[0]
            else:
                print('max_dist', max_dist, key)

    if len(matches) == 0:
        for exception in exceptions:
            if exception in key.lower():
                return None
        raise ValueError(f'Address not found: {details} at {key}')
    if len(matches) == 1:
        return matches[0]
    raise ValueError(f'Multiple address locations found:{details} at {key} {matches}')


def geocode(df, exceptions=None):
    for index, row in df.iterrows():
        addy = lookup(address_locations, key=row['Pickup Address'], details=row.get('Pickup Address Details', '?'),
                      exceptions=exceptions)
        if addy is not None:
            df.loc[index, 'Pickup Lat'] = addy['location']['lat']
            df.loc[index, 'Pickup Lng'] = addy['location']['lng']
            df.loc[index, 'Pickup addy'] = addy['formatted_address']

        addy = lookup(address_locations, key=row['Dropoff Address'], details=row.get('Dropoff Address Details', '?'),
                      exceptions=exceptions)
        if addy is not None:
            df.loc[index, 'Dropoff Lat'] = addy['location']['lat']
            df.loc[index, 'Dropoff Lng'] = addy['location']['lng']
            df.loc[index, 'Dropoff addy'] = addy['formatted_address']
    return df


In [ ]:
df = geocode(pd.read_csv(metroflex_data / 'metroflex-2025-06-trip-report-cleaned.csv'))
df
# https://gis.stackexchange.com/questions/478557/heatmap-using-latitude-and-longitude-coordinates

In [ ]:
{
    'max_pickup_lat': df['Pickup Lat'].max(),
    'max_dropoff_lat': df['Dropoff Lat'].max(),

    'max_pickup_lng': df['Pickup Lng'].max(),
    'max_dropoff_lng': df['Dropoff Lng'].max(),

    'min_pickup_lat': df['Pickup Lat'].min(),
    'min_dropoff_lat': df['Dropoff Lat'].min(),

    'min_pickup_lng': df['Pickup Lng'].min(),
    'min_dropoff_lng': df['Dropoff Lng'].min(),
}

In [ ]:
import math
from io import BytesIO
from PIL import Image
import requests
from itertools import product


class LonLatBounds:
    def __init__(self, left, top, right, bottom):
        # lon
        self.left = left
        self.right = right
        # lat
        self.top = top
        self.bottom = bottom


class PixelPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class MapTileIndex:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class OpenLayersClient:
    TILE_SIZE = 256
    URL = "https://tile.openstreetmap.org/{z}/{x}/{y}.png".format

    def lon_lat_to_pixels(self, lon, lat, zoom):
        """convert gps coordinates to web mercator"""
        r = math.pow(2, zoom) * self.TILE_SIZE
        lat = math.radians(lat)

        x = int((lon + 180.0) / 360.0 * r)
        y = int((1.0 - math.log(math.tan(lat) + (1.0 / math.cos(lat))) / math.pi) / 2.0 * r)

        return PixelPoint(x, y)

    def pixels_to_map_tile_index(self, pixel_x, pixel_y):
        return MapTileIndex(
            x=int(pixel_x / self.TILE_SIZE),
            y=int(pixel_y / self.TILE_SIZE),
        )

    def image_at(self, index_x, index_y, zoom):
        url = self.URL(x=index_x, y=index_y, z=zoom)

        with requests.get(url, headers={
            'User-Agent': 'Roanoke Transit Hobby Project - Justin Bangerter',
        }) as resp:
            resp.raise_for_status()
            return Image.open(BytesIO(resp.content))

    def image_at_lon_lat(self, lon, lat, zoom):
        px_point = self.lon_lat_to_pixels(lon, lat, zoom)
        tile_index = self.pixels_to_map_tile_index(px_point.x, px_point.y)
        return self.image_at(tile_index.x, tile_index.y, zoom)

    def tile_ct_for_bounds(self, bounds: LonLatBounds, zoom: int):
        p0 = self.lon_lat_to_pixels(bounds.left, bounds.top, zoom)
        p1 = self.lon_lat_to_pixels(bounds.right, bounds.bottom, zoom)

        x0_tile, y0_tile = int(p0.x / self.TILE_SIZE), int(p0.y / self.TILE_SIZE)
        x1_tile, y1_tile = math.ceil(p1.x / self.TILE_SIZE), math.ceil(p1.y / self.TILE_SIZE)

        tile_ct = (x1_tile - x0_tile) * (y1_tile - y0_tile)
        return tile_ct

    def stitched_tiles_for_bounds(self, bounds: LonLatBounds, zoom: int, max_tiles=50):
        tile_ct = self.tile_ct_for_bounds(bounds, zoom)
        if tile_ct > max_tiles:
            raise ValueError(f'tile_ct {tile_ct} > max_tiles {max_tiles}')
        p0 = self.lon_lat_to_pixels(bounds.left, bounds.top, zoom)
        p1 = self.lon_lat_to_pixels(bounds.right, bounds.bottom, zoom)

        x0_tile, y0_tile = int(p0.x / self.TILE_SIZE), int(p0.y / self.TILE_SIZE)
        x1_tile, y1_tile = math.ceil(p1.x / self.TILE_SIZE), math.ceil(p1.y / self.TILE_SIZE)

        # add tiles to base image
        img = Image.new('RGB', (
            (x1_tile - x0_tile) * self.TILE_SIZE,
            (y1_tile - y0_tile) * self.TILE_SIZE))

        for x_tile, y_tile in product(range(x0_tile, x1_tile), range(y0_tile, y1_tile)):
            tile_img = self.image_at(x_tile, y_tile, zoom)

            img.paste(
                im=tile_img,
                box=((x_tile - x0_tile) * self.TILE_SIZE, (y_tile - y0_tile) * self.TILE_SIZE))

        # crop
        x0 = x0_tile * self.TILE_SIZE
        y0 = y0_tile * self.TILE_SIZE
        img = img.crop((
            p0.x - x0,  # left
            p0.y - y0,  # top
            p1.x - x0,  # right
            p1.y - y0,  # bottom
        ))

        return img


In [ ]:
zoom = 13
bounds = LonLatBounds(
    left=-80.11,
    right=-79.86,
    top=37.35,
    bottom=37.21,
)

In [ ]:
ol_client = OpenLayersClient()
img = ol_client.stitched_tiles_for_bounds(bounds, zoom)
plt.imshow(img)
plt.show()

In [ ]:
import matplotlib.colors as colors
import shapely as shp
from shapely.geometry import Point


def plot_heat_map(df, bounds, map_image, direction, month, year, heatmap_step):
    img = map_image
    trip_point_type = direction

    month_name = [
        'January','February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November',
        'December'
    ][int(month) - 1]

    plt.close()

    # make values for a typical coordinate grid with a specified step
    step = heatmap_step
    lons_full = np.arange(bounds.left, bounds.right + step, step)
    lats_full = np.arange(bounds.top, bounds.bottom + step, -step)

    # get values for a special grid, representing central points of cells in the normal grid
    halfstep = step / 2
    lons = lons_full[:-1] + halfstep
    lats = lats_full[:-1] - halfstep

    # make a 2D array to keep square polygons corresponding to those grid cells
    squares = [[None for col in range(len(lons))] for row in range(len(lats))]
    squares_shape = np.asarray(squares).shape

    # make shapely polygons and store them in that array
    for row, col in np.ndindex(squares_shape):
        # longitudes correspond to columns, latitudes to rows
        lon = lons[col]
        lat = lats[row]
        # define vertices of a square polygon
        bottom_left = (lon - halfstep, lat - halfstep)
        bottom_right = (lon + halfstep, lat - halfstep)
        top_right = (lon + halfstep, lat + halfstep)
        top_left = (lon - halfstep, lat + halfstep)
        coords = (bottom_left, bottom_right, top_right, top_left, bottom_left)
        # make a polygon
        squares[row][col] = shp.Polygon(coords)

    # make a numpy array (of the same shape as that 2D polygon array)
    # where each element keeps the number pickups in the destination
    trip_count = np.zeros(squares_shape)
    for index, df_entry in df.iterrows():
        # each row is a set of line strings representing routes
        lat = df_entry[f'{trip_point_type} Lat']
        lng = df_entry[f'{trip_point_type} Lng']
        point = Point(lng, lat)
        # add passenger and guests to count
        for row, col in np.ndindex(squares_shape):
            square = squares[row][col]
            if point.within(square):
                trip_count[row, col] += 1  #+ int(df_entry['Guests'])

    # get coastlines and countries
    # countries_file = gpd.datasets.get_path('naturalearth_lowres')
    # countries_gdf = gpd.read_file(countries_file)

    # show everything on one plot
    fig, ax = plt.subplots(figsize=(15, 8))
    plt.title(f'Roanoke Metroflex: Distribution of Metroflex {trip_point_type} - {month_name}, {year}')
    # show coastlines and countries
    # countries_gdf.plot(ax=ax, facecolor='none', edgecolor='grey')

    # make bounds for 10 sections of a colour bar
    # cmap = cm.OrRd
    from matplotlib.colors import LinearSegmentedColormap

    cmap = LinearSegmentedColormap.from_list('', [
        'white',
        'darkblue',
        'darkgreen',
        'yellow',
        'darkorange',
        'darkred',
    ])

    trips_min = trip_count.min()
    trips_max = trip_count.max()
    lin_bounds = np.linspace(trips_min, trips_max + 1, 11)
    norm = colors.BoundaryNorm(lin_bounds, ncolors=cmap.N)

    extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)
    ax.imshow(img, extent=extent)

    # show squares, whose colour represents the trip count
    extent = (min(lons_full), max(lons_full), min(lats_full), max(lats_full))
    im = ax.imshow(trip_count, cmap=cmap, alpha=0.5, extent=extent, norm=norm)

    # add a color bar
    cb = plt.colorbar(im)
    cb.set_label('Number of Trips')
    ax.set_xlabel('Longitude (decimal degrees)')
    ax.set_ylabel('Latitude (decimal degrees)')

    return plt


In [ ]:

for year, month in [
    ('2025', '02',),
    ('2025', '03',),
    ('2025', '06',),
]:
    df = geocode(pd.read_csv(metroflex_data / f'metroflex-{year}-{month}-trip-report-cleaned.csv'))
    df.to_csv(output_dir / f'metroflex-{year}-{month}-geocoded.csv')

    for direction in ['Pickup', 'Dropoff']:
        for step in [.005, .0025]:
            plt = plot_heat_map(df, bounds, map_image=img, direction=direction, month=month, year=year, heatmap_step=step)
            plt.savefig(output_dir / f'metroflex-{year}-{month}-heat-map-{direction.lower()}-resolution-{step}.png')

In [ ]:
df

In [ ]:
import math
import ipyleaflet.basemaps
import ipywidgets

from transit import vector_2d


def create_map(df):
    ipywidgets.widgets.Widget.close_all()
    ipyleaflet.Map().clear()
    lat = (df['Pickup Lat'].mean() + df['Dropoff Lat'].mean()) / 2
    lng = (df['Pickup Lng'].mean() + df['Dropoff Lng'].mean()) / 2
    pymap = ipyleaflet.Map(
        center=(lat, lng), zoom=13, min_zoom=1, max_zoom=20,
        scroll_wheel_zoom=True,
        layout=ipywidgets.Layout(width='100%', min_height='800px'),
    )
    return pymap

for year, month in [
    # ('2025', '02',),
    # ('2025', '03',),
    ('2025', '06',),
]:
    df = geocode(pd.read_csv(metroflex_data / f'metroflex-{year}-{month}-trip-report-cleaned.csv'))
    pymap = create_map(df)

    for i, row in df.iterrows():
        p0 = np.array((row['Pickup Lat'], row['Pickup Lng']))
        if np.isnan(p0[0]) or np.isnan(p0[1]):
            continue
        p1 = np.array((row['Dropoff Lat'], row['Dropoff Lng']))
        if np.isnan(p1[0]) or np.isnan(p1[1]):
            continue
        pts = vector_2d.add_arrowhead_points(p0, p1, size=.0005)
        
        line = ipyleaflet.Polyline(
            locations = [list(pt) for pt in pts],
            color = "black",
            weight = 1,
            fill = False,
        )
        pymap.add(line)
    pymap.save(output_dir / f'metroflex-vectors-{year}-{month}.html')

pymap